# Code Generator tool - week 3

A code generator tool that can convert python code to various languages. You can also choose various models to do the conversion

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
languages = {
    'Javascript': 'js',
    'C++': 'cpp',
    'Ruby': 'rb',
}

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

In [ ]:
openai = OpenAI()
def stream_gpt(prompt,system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
def stream_claude(prompt, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
      ]
    stream = anthropic.chat.completions.create(
        model='claude-sonnet-4-5-20250929',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_answer(prompt, model, language):
    system_prompt = f"""You are a technical assistant, help me convert python code to {language}. Only return the code, no explanations."""
    if model=="GPT":
        result = stream_gpt(prompt, system_prompt)
    elif model=="Claude":
        result = stream_claude(prompt, system_prompt)
    else:
        raise ValueError("Unknown model.")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Enter your question here:", info="Enter a message for GPT-4.1-mini", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
lang_selector = gr.Dropdown(list(languages.keys()), label="Select programming language", value="Javascript")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_answer,
    title="GPT", 
    inputs=[message_input, model_selector, lang_selector],
    outputs=[message_output], 
    flagging_mode="never",
)
view.launch()